# 3 · Evaluación de los cuatro enfoques

(a) LLM libre · (b) LLM estructurado con referencias · (c) (b) + verificación y reintento · (d) plantilla determinista. Validación (5 informes, 2023) para ajustar el prompt; prueba (12 informes, 2024, otros países) corrida una sola vez con la versión elegida. Todo sale de las respuestas grabadas: este notebook no necesita Ollama.

In [1]:
import sys, os
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
import json
res = json.loads(open('reports/resultados.json', encoding='utf-8').read())
print('Versión de prompt elegida en validación:', res['version_prompt_elegida'])
pd.read_csv('reports/resumen_enfoques.csv')[['conjunto', 'enfoque', 'version', 'informes', 'cifras', 'tasa_alucinacion', 'tasa_alucinacion_ic', 'cobertura', 'cobertura_ic', 'informes_sin_error', 'segundos_por_informe']].round(3)

Versión de prompt elegida en validación: v1


,conjunto,enfoque,version,informes,cifras,tasa_alucinacion,tasa_alucinacion_ic,cobertura,cobertura_ic,informes_sin_error,segundos_por_informe
0,validacion,estructurado,v1,5,34,0.000,0.0000–0.0000,0.600,0.4000–0.8000,1.000,126.414
1,validacion,verificado,v1,5,34,0.000,0.0000–0.0000,0.600,0.4000–0.8000,1.000,126.414
2,validacion,estructurado,v2,5,24,0.042,0.0000–0.1500,0.800,0.7500–0.9000,0.600,130.857
3,validacion,verificado,v2,5,40,0.025,0.0000–0.1034,0.750,0.7500–0.7500,0.800,230.148
4,validacion,libre,v1,5,41,0.024,0.0000–0.0639,0.650,0.5500–0.7500,0.800,81.862
5,validacion,plantilla,-,5,120,0.000,0.0000–0.0000,0.900,0.8000–1.0000,1.000,0.012
6,prueba,libre,v1,12,101,0.040,0.0106–0.0680,0.667,0.5625–0.7708,0.667,55.900
7,prueba,estructurado,v1,12,99,0.061,0.0000–0.1405,0.812,0.6875–0.9375,0.833,144.397
8,prueba,verificado,v1,12,97,0.031,0.0000–0.0789,0.812,0.6875–0.9375,0.917,230.114
9,prueba,plantilla,-,12,288,0.000,0.0000–0.0000,0.646,0.5417–0.7292,1.000,0.012


![Alucinación y cobertura](../reports/figures/03_alucinacion_y_cobertura.png)

## Diferencias pareadas en prueba (mismos 12 informes, bootstrap por informe)

In [2]:
pd.read_csv('reports/comparaciones_pareadas.csv').round(3)

,a,b,metrica,diferencia,ic_inf,ic_sup
0,verificado,estructurado,tasa_alucinacion,-0.030,-0.069,0.000
1,verificado,estructurado,cobertura,0.000,0.000,0.000
2,estructurado,libre,tasa_alucinacion,0.021,-0.052,0.109
3,estructurado,libre,cobertura,0.146,0.062,0.250
4,verificado,libre,tasa_alucinacion,-0.009,-0.063,0.051
5,verificado,libre,cobertura,0.146,0.062,0.250
6,plantilla,verificado,tasa_alucinacion,-0.031,-0.080,0.000
7,plantilla,verificado,cobertura,-0.167,-0.292,-0.042


## Qué tipo de errores comete cada enfoque

In [3]:
cif = pd.read_csv('reports/detalle_cifras.csv')
p = cif[cif.conjunto == 'prueba']
p.groupby(['enfoque', 'estado']).size().unstack(fill_value=0)

estado,correcta,incorrecta
enfoque,,
estructurado,93,6
libre,97,4
plantilla,288,0
verificado,94,3


In [4]:
p[p.estado != 'correcta'][['enfoque', 'pais', 'seccion', 'cifra', 'estado', 'valor_real', 'oracion']].head(25)

,enfoque,pais,seccion,cifra,estado,valor_real,oracion
301,libre,BOL,linea_3,15,incorrecta,4.275 USD,- **PIB y PIB per cápita**: Bolivia registra u...
325,libre,COL,linea_8,"4,1",incorrecta,"16,1 %",- **Déficit Comercial**: Las importaciones sup...
353,libre,MEX,linea_7,segundo país con mayor,incorrecta,puesto 10 de 18,- **Crecimiento Económico Moderado**: A pesar ...
385,libre,URY,linea_4,18,incorrecta,23.379 USD,- **Desempeño Económico**: El PIB per cápita a...
435,estructurado,HND,hallazgo_1,16,incorrecta,"3,6 %","El PIB de Honduras creció un 7,6% en 2024, sup..."
436,estructurado,HND,hallazgo_1,15,incorrecta,"3,6 %","El PIB de Honduras creció un 7,6% en 2024, sup..."
439,estructurado,HND,hallazgo_2,primer puesto,incorrecta,puesto 2 de 18,"Las remesas a Honduras representaron el 25,7% ..."
441,estructurado,HND,hallazgo_3,13,incorrecta,"6,1 %","El desempleo en Honduras disminuyó a 4,9%, mej..."
442,estructurado,HND,hallazgo_3,12,incorrecta,"6,1 %","El desempleo en Honduras disminuyó a 4,9%, mej..."
491,estructurado,URY,hallazgo_2,18,incorrecta,"3,3 %",El crecimiento del PIB real supera la mediana ...


![Tipos de error](../reports/figures/04_tipos_de_error.png)

## Reintentos de (c)

In [5]:
pd.read_csv('reports/reintentos.csv').query("conjunto == 'prueba'")

,conjunto,pais,version,intento,n_cifras,errores,json_valido
13,prueba,ECU,v1,0,12,0,True
14,prueba,BOL,v1,0,6,0,True
15,prueba,BOL,v1,1,6,0,True
16,prueba,BRA,v1,0,5,0,True
17,prueba,COL,v1,0,6,0,True
18,prueba,CRI,v1,0,5,0,True
19,prueba,DOM,v1,0,5,0,True
20,prueba,HND,v1,0,19,5,True
21,prueba,HND,v1,1,19,4,True
22,prueba,HND,v1,2,19,3,True


![Reintentos](../reports/figures/05_reintentos.png)

## Ajuste del prompt (solo validación)

![Prompt](../reports/figures/06_mejora_prompt_validacion.png)

## Auditoría manual del verificador y legibilidad

In [6]:
aud = pd.read_csv('data/auditoria_verificador.csv')
pd.crosstab(aud.estado_verificador, aud.juicio_humano, margins=True)

juicio_humano,correcta,incorrecta,sin_fuente,All
estado_verificador,,,,
correcta,38,1,1,40
incorrecta,3,6,4,13
All,41,7,5,53


In [7]:
pd.read_csv('data/rubrica_legibilidad.csv').merge(
    pd.DataFrame(json.loads(open('data/rubrica_mapa.json', encoding='utf-8').read())).T.reset_index()
    .rename(columns={'index': 'id'}), on='id').groupby('enfoque')[['claridad', 'concision', 'utilidad', 'tono']].mean().round(2)

,claridad,concision,utilidad,tono
enfoque,,,,
estructurado,2.83,3.17,2.33,2.83
libre,4.00,3.83,3.00,4.00
plantilla,4.00,4.00,3.50,3.00
verificado,2.83,3.17,2.50,2.83


![Legibilidad](../reports/figures/07_legibilidad_rubrica.png)

![Tiempo](../reports/figures/08_tiempo_por_informe.png)